# 2. How do spring differences reach the nodes?

In Energy 01, six nodes formed a chain with five springs. We fixed $z_0=1$
and $z_5=3$ and minimized the energy. Here we start from the same uneven
configuration $[1,1.25,1.5,2,2.25,3]$ and work out how individual springs
contribute to the energy gradient at the nodes.

The signed difference of spring $i$ is $r_i=z_{i+1}-z_i$. Each row of
$D\in\mathbb R^{5\times6}$ subtracts the left node and adds the right node.
Give $D$ to `AffineResponse` with its default zero offset. This gives the
response $r(z)=Dz$, whose Jacobian $J_r=D$ is constant.

| Quantity | Where it lives | Length |
| --- | --- | --- |
| Positions $z$, perturbation $v$ | Nodes | 6 |
| Differences $r(z)$, change $J_rv$ | Springs | 5 |
| Weights $w$ | Springs | 5 |
| Pulled-back weights $J_r^\top w$, energy gradient $\nabla E$ | Nodes | 6 |

In [ ]:
#include <cmath>
#include <iostream>
#include <stdexcept>
#include <vector>

#include "litepgo/energy/response/affine_response.hpp"
#include "litepgo/energy/scalar/quadratic_energy.hpp"

static_assert(__cplusplus >= 202002L);
using namespace litepgo;

constexpr int n = 6;
Vec z = (Vec(n) << 1.0, 1.25, 1.5, 2.0, 2.25, 3.0).finished();
constexpr double tolerance = 1e-10;

SpMat make_difference_matrix() {
    std::vector<Triplet> entries;
    for (int i = 0; i < n - 1; ++i) {
        entries.emplace_back(i, i, -1.0);
        entries.emplace_back(i, i + 1, 1.0);
    }
    SpMat matrix(n - 1, n);
    matrix.setFromTriplets(entries.begin(), entries.end());
    return matrix;
}

SpMat D = make_difference_matrix();
energy::AffineResponse response(D);
Vec r = response.evaluate(z)->value();
Vec expected_r = (Vec(n - 1) << 0.25, 0.25, 0.5, 0.25, 0.75).finished();

if (!(response.num_inputs() == n && response.num_outputs() == n - 1 &&
      (r - expected_r).cwiseAbs().maxCoeff() < tolerance)) {
    throw std::runtime_error("Spring response check failed");
}
std::cout << "Node positions " << z.transpose() << '\n'
          << "Spring differences " << r.transpose() << '\n';

## JVP · Move one node and read the changes in the springs

Choose the six-entry direction $v=e_2$ to move only node 2 to the right.
Before running the cell, predict which two spring differences change and
the sign of each change. The Jacobian-vector product (JVP)
$\texttt{eval\_jvp}(z,v)$ returns the five-entry vector $J_rv$. Since the
response is affine, it also equals the difference $r(z+v)-r(z)$ exactly.

In [ ]:
Vec v = Vec::Unit(n, 2);
Vec jvp = response.evaluate(z)->jvp(v);

if (!((jvp - D * v).cwiseAbs().maxCoeff() < tolerance &&
      (jvp - (response.evaluate(z + v)->value() - r)).cwiseAbs().maxCoeff() < tolerance)) {
    throw std::runtime_error("JVP check failed");
}
std::cout << "Node direction " << v.transpose() << '\n'
          << "Change in the springs " << jvp.transpose() << '\n';

Spring 1 ends at node 2 and gains one unit. Spring 2 starts at node 2 and
loses one unit.

## VJP · Send five spring weights back to the nodes

Give the springs weights $w=[1,2,4,1,3]$ and consider the scalar
$w^\top r(z)$. Spring $i$ contributes $-w_i$ to its left node and $+w_i$
to its right node. A shared node adds the contributions from both adjacent
springs. At node 2 the result is $+w_1-w_2=2-4=-2$.

The vector-Jacobian product (VJP) $\texttt{eval\_vjp}(z,w)$ returns these
weights as a six-entry column vector $J_r^\top w$. Check the same weighted
change two ways. The left dot product is in spring space. The right one
is in node space.

$$
w^\top(J_rv)=(J_r^\top w)^\top v.
$$

In [ ]:
Vec w = (Vec(n - 1) << 1.0, 2.0, 4.0, 1.0, 3.0).finished();
Vec vjp = response.evaluate(z)->vjp(w);
double spring_dot = w.dot(jvp);
double node_dot = vjp.dot(v);

if (!((vjp - D.transpose() * w).cwiseAbs().maxCoeff() < tolerance &&
      std::abs(spring_dot - node_dot) < tolerance)) {
    throw std::runtime_error("VJP or dot product check failed");
}
std::cout << "Spring weights " << w.transpose() << '\n'
          << "Weights at the nodes " << vjp.transpose() << '\n'
          << "Dot product in spring space " << spring_dot << '\n'
          << "Dot product in node space " << node_dot << '\n';

## Use the VJP to recover the energy gradient

The Energy 01 objective is $E(z)=\tfrac12 r(z)^\top r(z)$. The derivative
of $\tfrac12 r^\top r$ with respect to the five spring differences is $r$,
so use $w=r(z)$ in the VJP. The chain rule gives the six-entry gradient

$$
\nabla E(z)=J_r^\top r(z).
$$

Compare this with the gradient from `QuadraticEnergy` and with centered
finite differences of the scalar energy. For each node $i$, use
$(E(z+h e_i)-E(z-h e_i))/(2h)$ with $h=10^{-6}$. We check all six
partials, including those at the two endpoints. The fixed-end solve in
Energy 01 used only the four interior entries.

In [ ]:
SpMat H = D.transpose() * D;
energy::QuadraticEnergy chain(H);
Vec gradient_vjp = response.evaluate(z)->vjp(r);
Vec gradient_energy = chain.evaluate(z)->gradient();

Vec finite_difference_gradient(const Vec& positions, double h) {
    Vec gradient(n);
    for (int i = 0; i < n; ++i) {
        Vec shift = h * Vec::Unit(n, i);
        gradient[i] = (chain.evaluate(positions + shift)->value() - chain.evaluate(positions - shift)->value()) / (2 * h);
    }
    return gradient;
}

constexpr double h = 1e-6;
constexpr double fd_tolerance = 1e-8;
Vec gradient_fd = finite_difference_gradient(z, h);
double energy_error = (gradient_vjp - gradient_energy).cwiseAbs().maxCoeff();
double fd_error = (gradient_vjp - gradient_fd).cwiseAbs().maxCoeff();
if (!(energy_error < tolerance && fd_error < fd_tolerance)) {
    throw std::runtime_error("Energy gradient check failed");
}
std::cout << "Gradient from the response " << gradient_vjp.transpose() << '\n'
          << "Gradient from the energy " << gradient_energy.transpose() << '\n'
          << "Gradient from finite differences " << gradient_fd.transpose() << '\n'
          << "Maximum energy API error " << energy_error << '\n'
          << "Maximum finite difference error " << fd_error << '\n';

## Try it

Give spring 3 weight one and every other spring weight zero. Predict which
two nodes receive a nonzero contribution and what their signs are. Run the
cell, then try a different spring index from 0 to 4.

In [ ]:
{
    int spring = 3;
    Vec trial_weights = Vec::Unit(n - 1, spring);
    Vec trial_nodes = response.evaluate(z)->vjp(trial_weights);
    Vec expected_nodes = Vec::Zero(n);
    expected_nodes[spring] = -1.0;
    expected_nodes[spring + 1] = 1.0;

    if (!((trial_nodes - expected_nodes).cwiseAbs().maxCoeff() < tolerance)) {
        throw std::runtime_error("Single spring VJP check failed");
    }
    std::cout << "Spring weights " << trial_weights.transpose() << '\n'
              << "Weights at the nodes " << trial_nodes.transpose() << '\n';
}

<details>
<summary>Check your prediction</summary>

Spring 3 connects nodes 3 and 4. Its VJP is $[0,0,0,-1,1,0]$.
Changing the spring index moves the same two contributions along the chain.

</details>

**Checkpoint** You can follow a direction from six nodes to five springs with
a JVP, pull five weights back to six nodes with a VJP, and explain why choosing
the spring differences as weights gives the energy gradient.